# Module 4: AgentCore Memory -- Persistent Context

![Overview](../shared/img/04.drawio.png)

In this module you will give Aria a **memory** so she can remember who you are across conversations.

---

### What you will learn

| Topic | Details |
|---|---|
| **AgentCore Memory service** | The managed memory layer that ships with Bedrock AgentCore |
| **Short-term vs. long-term memory** | How STM (conversation buffer) and LTM (extracted knowledge) work together |
| **LTM extraction strategies** | Session summaries, user preferences, and semantic facts |
| **Strands integration** | How `AgentCoreMemorySessionManager` plugs into a Strands agent |

## Catch-up

The cell below checks that every resource from previous modules is in place. If you skipped a module or something broke, it will fix things automatically.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.ensure_ready import ensure_ready

config = ensure_ready("04")

---

## Understanding Memory

AgentCore Memory gives your agent the ability to **remember** information across sessions. It operates on two tiers:

### Short-term memory (STM)

When your agent sends conversation events to AgentCore Memory, the service stores them as **events**. These events capture each turn of the conversation. Your agent can retrieve these events to rehydrate its conversation history — useful if you want to resume a session later or carry on where you left off. Sending events to Memory is also what triggers the extraction of long-term memories (see below).

### Long-term memory (LTM)

Extracted knowledge that **persists across sessions**. As conversation events are sent to Memory during a session, the service runs extraction strategies in the background to distill the dialogue into durable knowledge. This extraction typically takes around a minute. The extracted memories can be retrieved within the same conversation if it runs long enough, but the primary purpose is to provide context in **future sessions**. When a new session starts, relevant LTM is retrieved and injected into the agent's context.

AgentCore Memory supports three LTM extraction strategies:

| Strategy | Class | Namespace | Purpose |
|---|---|---|---|
| **Session Summarizer** | `summaryMemoryStrategy` | `/summaries/{actorId}/{sessionId}` | Creates a concise summary of the conversation so future sessions can quickly recall what was discussed |
| **Preference Learner** | `userPreferenceMemoryStrategy` | `/preferences/{actorId}` | Detects and stores user preferences (e.g., "I prefer Python over Java") |
| **Fact Extractor** | `semanticMemoryStrategy` | `/facts/{actorId}` | Extracts factual statements (e.g., "I work at Acme Corp", "My name is Alex") |

Each strategy writes to its own **namespace** and the `{actorId}` placeholder is replaced at runtime with the authenticated user's identity, keeping each user's memories isolated.

> **Documentation:** [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html)

---

### Create the Memory resource

The `create_memory` API provisions a **memory resource** — think of it as creating a place where memories will be stored, not creating an actual memory of a fact. The memory resource is essentially a managed data store backed by DynamoDB.

You _can_ set up a memory resource without any LTM strategies at all. In that case, it simply stores conversation events (acting as a wrapper for DynamoDB). LTM strategies are optional add-ons that extract knowledge from those stored events.

Here we configure all three strategies so Aria gets the full memory experience:
- **SessionSummarizer** — Summarizes conversations when a session ends
- **PreferenceLearner** — Extracts user preferences ("I prefer dark mode")
- **FactExtractor** — Stores semantic facts ("Alex is a software engineer")

Each strategy has:
- A **name** and **description** for identification
- **Namespaces** — template paths that scope stored memories (e.g., per user, per session)

In [ ]:
import boto3
from botocore.exceptions import ClientError

region = "us-east-1"
client = boto3.client("bedrock-agentcore-control", region_name=region)

try:
    response = client.create_memory(
        name="AriaMemory",
        description="Memory for Aria personal assistant — conversation persistence and long-term user knowledge",
        eventExpiryDuration=90,  # Days before memory events expire
        memoryStrategies=[
            {
                "summaryMemoryStrategy": {
                    "name": "SessionSummarizer",
                    "description": "Summarizes conversation sessions for quick context retrieval",
                    "namespaces": ["/summaries/{actorId}/{sessionId}"],
                }
            },
            {
                "userPreferenceMemoryStrategy": {
                    "name": "PreferenceLearner",
                    "description": "Learns and stores user preferences across sessions",
                    "namespaces": ["/preferences/{actorId}"],
                }
            },
            {
                "semanticMemoryStrategy": {
                    "name": "FactExtractor",
                    "description": "Extracts and stores factual information from conversations",
                    "namespaces": ["/facts/{actorId}"],
                }
            },
        ],
    )
    memory_id = response["memory"]["id"]
    print(f"Memory created: {memory_id}")
    print(f"   Status: {response['memory'].get('status', 'CREATING')}")

except ClientError as e:
    if "already exists" in str(e):
        print("AriaMemory already exists -- looking it up...")
        paginator = client.get_paginator("list_memories")
        for page in paginator.paginate():
            for mem in page.get("memories", []):
                if mem["id"].startswith("AriaMemory"):
                    memory_id = mem["id"]
                    print(f"Found existing memory: {memory_id}")
                    break
    else:
        raise

### Wait for the Memory resource to become ACTIVE

Memory creation is asynchronous. We poll until the status reaches `ACTIVE`.

In [ ]:
import time

print(f"Waiting for memory {memory_id} to become ACTIVE...")
for i in range(30):
    resp = client.get_memory(memoryId=memory_id)
    memory = resp.get("memory", resp)
    status = memory.get("status", "UNKNOWN")
    print(f"  [{i*10}s] Status: {status}")
    if status == "ACTIVE":
        print("Memory is ACTIVE!")
        break
    if status in ("FAILED", "DELETE_FAILED"):
        print(f"Memory creation failed: {status}")
        break
    time.sleep(10)

### Save the Memory ID for later modules

We persist the memory ID to the shared config directory so that `ensure_ready()` and deployment helpers can find it in subsequent modules.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared import utils

utils.save_config("memory", {"memory_id": memory_id, "region": region})
print(f"Saved memory_id={memory_id}")

---
## Enable Tracing for Memory

Enable **Tracing** on the Memory resource so that memory operations (store, retrieve, extraction) appear in the AgentCore Observability dashboard.

1. Open the **Amazon Bedrock AgentCore** console and navigate to **Memory**
2. Select the **AriaMemory** resource
3. Scroll down to the **Tracing** section and click **Edit**

![Tracing section](../shared/img/tracing-01.png)

4. Toggle **Enable** on and click **Save**

![Enable tracing](../shared/img/tracing-02.png)

To see the full agent code, open [agent/main.py](agent/main.py) in a new tab.

---

## Deploy with Memory

We redeploy Aria with the `MEMORY_ID` environment variable pointing to the Memory resource we just created. The runtime will pick up the new agent code **and** the memory configuration.

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.deploy_agent import deploy

result = deploy(
    agent_dir="agent",
    env_vars={"MEMORY_ID": memory_id},
)
runtime_arn = result["runtime_arn"]

---

## Test Memory: Conversation 1

Let's have a conversation where we tell Aria some facts and preferences. Both messages use the **same session ID** so they share the conversation buffer (STM).

### Tell Aria about yourself

In [ ]:
import boto3, json, uuid
import sys; sys.path.insert(0, '..')
from shared import utils

data_client = boto3.client("bedrock-agentcore", region_name="us-east-1")

# Session 1: Tell Aria about yourself
session_1 = str(uuid.uuid4())
print(f"Session 1: {session_1[:16]}...")

response = data_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=session_1,
    payload=json.dumps({
        "prompt": "My name is Alex and I'm a software engineer. I prefer Python over Java.",
        "session_id": session_1,
    }).encode(),
)
utils.stream_sse_response(response["response"])

### Same session -- STM recall (conversation buffer)

Aria should recall your name and preferences because the first message is still in the conversation buffer.

In [ ]:
# Same session: STM recall (conversation buffer)
response = data_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=session_1,
    payload=json.dumps({
        "prompt": "What's my name and what language do I prefer?",
        "session_id": session_1,
    }).encode(),
)
utils.stream_sse_response(response["response"])

---

## Test Memory: New session (LTM recall)

Now we start a **brand-new session**. The conversation buffer is empty, so Aria can only recall facts if the LTM extraction strategies have processed the previous conversation and stored your preferences and facts.

> **Note:** LTM extraction happens in the background as conversation events are sent to Memory. It typically takes around a minute for memories to be formed and saved. If the new session returns no memories, wait 30--60 seconds and try again.

In [ ]:
# NEW session: LTM recall (cross-session)
session_2 = str(uuid.uuid4())
print(f"Session 2 (new): {session_2[:16]}...")

response = data_client.invoke_agent_runtime(
    agentRuntimeArn=runtime_arn,
    runtimeSessionId=session_2,
    payload=json.dumps({
        "prompt": "What do you remember about me?",
        "session_id": session_2,
    }).encode(),
)
utils.stream_sse_response(response["response"])

If everything is working, Aria should recall that your name is Alex, that you are a software engineer, and that you prefer Python over Java -- even though this is a completely new session.

---

## How it works

Here is the end-to-end flow:

- The session starts, or restarts. AgentCoreMemorySessionManager retrieves message events if they exist for this session. 
- User enters a message. 
- AgentCoreMemorySessionManager performs a LTM query of AgentCore Memory and returns semantically related memories. They get prepended to message before sending to agentic loop.
- Agent processes messages.
- All messages, including generated messages are sent via AgentCoreMemorySessionManager to AgentCore Memory as events.
- AgentCore Memory runs extraction strategies in the background to distill the conversation into durable knowledge. The extracted memories are stored in the configured namespaces.

The retrieval configs use different thresholds because:

- **Preferences** (`relevance_score=0.7`): Only highly relevant preferences should influence responses
- **Facts** (`relevance_score=0.3`, `top_k=10`): Cast a wider net to bring in background knowledge
- **Summaries** (`relevance_score=0.5`): Balance between relevance and coverage of past conversations

> **Go deeper:** [AgentCore Memory documentation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html) — includes custom memory strategies, memory management APIs, and advanced retrieval configuration

---

## What's next

Memory gives Aria a brain -- she can remember who you are and what you care about. But right now she can only **chat**. She cannot interact with external systems on your behalf.

In **Module 5: Gateway & Identity**, we will connect Aria to a Task Management API through AgentCore Gateway, and use JWT-based identity so each user's tasks stay private.

---

## Progress

In [ ]:
import sys; sys.path.insert(0, '..')
from shared.progress import show

show("04")

---

**Next up: [Module 5 -- Connect Aria to the Gateway](../05-gateway-identity/notebook.ipynb)**